## Homework Sesi 7 — Retail Dataset Automation & Cleaning

Membersihkan dataset transaksi retail mentah memakai Class `DataCleaner` dari hands-on Sesi 7 (pipeline data logistik), yang dimodifikasi untuk dataset retail. Logikanya lalu dipindah ke script `clean_retail.py` supaya bisa dijalankan sekali panggil.

Alur notebook:
1. Load & eksplorasi data mentah
2. Modifikasi Class `DataCleaner` dari hands-on
3. Uji coba setiap method secara interaktif
4. Validasi & ekspor `dataset_retail_clean.csv`


### 1. Load Data Mentah

In [1]:
from pathlib import Path

import pandas as pd

# Brief menyebut dataset_retail_homework.csv, file yang dibagikan bernama dataset_retail.csv
INPUT_CANDIDATES = ["dataset_retail_homework.csv", "dataset_retail.csv"]
input_path = next(Path(nama) for nama in INPUT_CANDIDATES if Path(nama).exists())

df_raw = pd.read_csv(input_path)
print(f"File  : {input_path}")
print(f"Ukuran: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")
df_raw.head(10)

File  : dataset_retail.csv
Ukuran: 108 baris x 8 kolom


,id_transaksi,nama_produk,kategori,jumlah_beli,harga_satuan,metode_bayar,kota_pembeli,tanggal_transaksi
0,INV-5000,Laptop Asus ROG,Aksesoris,5.0,20000000.0,KARTU KREDIT,NaN,NaN
1,INV-5001,Keyboard Mechanical,Aksesoris,2.0,NaN,TRANSFER BANK,Surabaya,2026-02-01
2,INV-5002,KEYBOARD MECHANICAL,gadget,10.0,NaN,KARTU KREDIT,bandung,NaN
3,INV-5003,KEYBOARD MECHANICAL,AKSESORIS,3.0,150000.0,e-wallet,JAKARTA,NaN
4,INV-5004,IPHONE 15 PRO,elektronik,2.0,NaN,Transfer Bank,Medan,2026/02/03
5,INV-5005,Monitor 24 Inch,gadget,3.0,50000.0,Kartu Kredit,NaN,2026-02-02
6,INV-5006,mouse wireless,elektronik,1.0,2500000.0,TRANSFER BANK,JAKARTA,2026-02-01
7,INV-5007,Charger Type C,elektronik,3.0,2500000.0,Transfer Bank,Jakarta,2026/02/03
8,INV-5008,Headset Gaming,elektronik,1.0,15000000.0,TRANSFER BANK,Jakarta,2026-02-02
9,INV-5009,KEYBOARD MECHANICAL,gadget,1.0,2500000.0,COD,bandung,2026-02-02


In [2]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_transaksi       108 non-null    str    
 1   nama_produk        108 non-null    str    
 2   kategori           108 non-null    str    
 3   jumlah_beli        96 non-null     float64
 4   harga_satuan       92 non-null     float64
 5   metode_bayar       108 non-null    str    
 6   kota_pembeli       90 non-null     str    
 7   tanggal_transaksi  86 non-null     str    
dtypes: float64(2), str(6)
memory usage: 6.9 KB


### 2. Eksplorasi Masalah Data

**a. Missing values per kolom**

In [3]:
df_raw.isna().sum().to_frame("jumlah_nan")

,jumlah_nan
id_transaksi,0
nama_produk,0
kategori,0
jumlah_beli,12
harga_satuan,16
metode_bayar,0
kota_pembeli,18
tanggal_transaksi,22


**b. Variasi penulisan pada kolom teks** — teks yang sama ditulis dengan huruf besar/kecil dan spasi yang berbeda-beda. `repr()` dipakai supaya spasi di awal/akhir terlihat.

In [4]:
KOLOM_TEKS = ["nama_produk", "kategori", "metode_bayar", "kota_pembeli"]

for col in KOLOM_TEKS:
    nilai_unik = df_raw[col].dropna().unique()
    print(f"{col} ({len(nilai_unik)} nilai unik):")
    print("   ", [repr(v) for v in sorted(nilai_unik)])

nama_produk (10 nilai unik):
    ["' Laptop Asus ROG '", "' Monitor 24 Inch '", "' Mouse Wireless '", "' mouse wireless '", "'Charger Type C'", "'Headset Gaming'", "'IPHONE 15 PRO '", "'KEYBOARD MECHANICAL'", "'Keyboard Mechanical'", "'iPhone 15 Pro'"]
kategori (6 nilai unik):
    ["'AKSESORIS'", "'Aksesoris'", "'Elektronik'", "'Gadget'", "'elektronik '", "'gadget'"]
metode_bayar (7 nilai unik):
    ["'COD'", "'E-Wallet'", "'KARTU KREDIT'", "'Kartu Kredit'", "'TRANSFER BANK'", "'Transfer Bank'", "'e-wallet '"]
kota_pembeli (5 nilai unik):
    ["'JAKARTA '", "'Jakarta'", "'Medan '", "'Surabaya'", "'bandung'"]


**c. Baris duplikat persis**

In [5]:
print(f"Jumlah baris duplikat: {df_raw.duplicated().sum()}")
# keep=False -> tampilkan baris asli dan salinannya berdampingan
df_raw[df_raw.duplicated(keep=False)].sort_values("id_transaksi")

Jumlah baris duplikat: 8


,id_transaksi,nama_produk,kategori,jumlah_beli,harga_satuan,metode_bayar,kota_pembeli,tanggal_transaksi
3,INV-5003,KEYBOARD MECHANICAL,AKSESORIS,3.0,150000.0,e-wallet,JAKARTA,NaN
102,INV-5003,KEYBOARD MECHANICAL,AKSESORIS,3.0,150000.0,e-wallet,JAKARTA,NaN
12,INV-5012,Headset Gaming,elektronik,1.0,2500000.0,Kartu Kredit,bandung,2026-02-02
105,INV-5012,Headset Gaming,elektronik,1.0,2500000.0,Kartu Kredit,bandung,2026-02-02
13,INV-5013,Charger Type C,AKSESORIS,NaN,15000000.0,KARTU KREDIT,bandung,2026/02/03
107,INV-5013,Charger Type C,AKSESORIS,NaN,15000000.0,KARTU KREDIT,bandung,2026/02/03
14,INV-5014,Mouse Wireless,Elektronik,5.0,20000000.0,Transfer Bank,bandung,04-02-2026
104,INV-5014,Mouse Wireless,Elektronik,5.0,20000000.0,Transfer Bank,bandung,04-02-2026
15,INV-5015,Charger Type C,gadget,2.0,50000.0,COD,Medan,2026-02-01
101,INV-5015,Charger Type C,gadget,2.0,50000.0,COD,Medan,2026-02-01


Ringkasan temuan:
- 4 kolom teks punya spasi berlebih dan kapitalisasi tidak seragam (contoh: `' mouse wireless '`, `'MOUSE WIRELESS'`, `'Mouse Wireless'`).
- Ada 8 baris yang identik persis.
- `jumlah_beli` punya nilai kosong (NaN).
- `kota_pembeli` juga punya NaN — ini penting untuk modifikasi pertama di bawah.

### 3. Modifikasi Class `DataCleaner` dari Hands-on

Struktur class tetap sama dengan versi hands-on: CSV dibaca di `__init__`, nama method `bersihkan_teks`, `hapus_duplikat`, `isi_nilai_kosong`, `simpan_hasil`, setiap method mengembalikan `self` untuk *method chaining*, dan dilengkapi type hinting + docstring.

Yang dimodifikasi (ditandai komentar `MODIFIKASI` di kode):

| Method | Versi hands-on | Modifikasi | Alasan |
|---|---|---|---|
| `bersihkan_teks` | `.astype(str).str.strip().str.title()` | Tanpa `.astype(str)` | Di pandas 2.x, `astype(str)` mengubah NaN menjadi teks `"nan"` → `"Nan"`. Pada data retail, 18 baris `kota_pembeli` yang kosong akan terlihat seperti kota bernama "Nan". Accessor `.str` sudah otomatis melewati NaN. |
| `bersihkan_teks` | Kolom yang tidak ada dilewati diam-diam | Tampilkan peringatan | Supaya salah ketik nama kolom langsung ketahuan. |
| `hapus_duplikat` | `drop_duplicates()` | `+ reset_index(drop=True)` | Index kembali berurutan setelah baris dibuang. |
| `isi_nilai_kosong` | Hanya `fillna` | Laporkan jumlah sel yang diisi + ubah ke `int` | Kolom berisi NaN dibaca pandas sebagai float (`5.0`), padahal jumlah barang selalu bilangan bulat. |
| `simpan_hasil` | — | Tampilkan jumlah baris yang disimpan | Laporan akhir lebih informatif. |

In [6]:
class DataCleaner:
    """
    Class otomatisasi pembersihan data CSV secara praktis dan terstruktur.

    Attributes:
        file_path (str): Path lokasi file CSV mentah.
        df (pd.DataFrame): DataFrame Pandas tempat data disimpan dan diproses.
    """

    def __init__(self, file_path: str):
        """Inisialisasi class dan membaca file CSV."""
        self.file_path: str = str(file_path)
        self.df: pd.DataFrame = pd.read_csv(file_path)
        print(f"✓ Berhasil membaca file '{Path(file_path).name}' dengan {len(self.df)} baris.")

    def bersihkan_teks(self, daftar_kolom: list[str]) -> "DataCleaner":
        """Menghilangkan spasi berlebih dan merapikan format teks (Title Case)."""
        for col in daftar_kolom:
            if col in self.df.columns:
                # MODIFIKASI: tanpa .astype(str) supaya NaN tetap NaN (bukan teks "Nan")
                self.df[col] = self.df[col].str.strip().str.title()
            else:
                # MODIFIKASI: beri peringatan kalau nama kolom salah ketik
                print(f"⚠ Kolom '{col}' tidak ditemukan, dilewati.")
        print(f"✓ Kolom teks {daftar_kolom} telah dibersihkan.")
        return self

    def hapus_duplikat(self) -> "DataCleaner":
        """Menghapus seluruh baris yang terduplikasi persis."""
        jumlah_awal = len(self.df)
        # MODIFIKASI: reset_index supaya index kembali berurutan setelah baris dibuang
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        jumlah_terhapus = jumlah_awal - len(self.df)
        print(f"✓ Berhasil menghapus {jumlah_terhapus} baris duplikat.")
        return self

    def isi_nilai_kosong(self, kolom_angka: str, nilai_default: float = 0.0) -> "DataCleaner":
        """Mengisi nilai yang hilang/kosong pada kolom numerik."""
        if kolom_angka in self.df.columns:
            # MODIFIKASI: hitung dulu berapa sel yang diisi, untuk laporan
            jumlah_kosong = self.df[kolom_angka].isna().sum()
            self.df[kolom_angka] = self.df[kolom_angka].fillna(nilai_default)
            # MODIFIKASI: jumlah barang selalu bilangan bulat, jadi kembalikan ke int
            if (self.df[kolom_angka] % 1 == 0).all():
                self.df[kolom_angka] = self.df[kolom_angka].astype(int)
            print(
                f"✓ {jumlah_kosong} nilai kosong pada kolom '{kolom_angka}' "
                f"diisi dengan {nilai_default}."
            )
        return self

    def simpan_hasil(self, output_path: str) -> "DataCleaner":
        """Menyimpan dataframe yang sudah bersih ke file CSV baru."""
        self.df.to_csv(output_path, index=False)
        print(f"✓ Data bersih berhasil disimpan ke: '{Path(output_path).name}' ({len(self.df)} baris).")
        return self

### 4. Uji Coba Method Secara Interaktif

Setiap method dijalankan satu per satu dulu (bukan langsung *chaining*) supaya efeknya bisa dicek di tiap langkah.

**Langkah 1 - Inisialisasi objek (sekaligus load data)**

In [7]:
cleaner = DataCleaner(input_path)
cleaner.df.head()

✓ Berhasil membaca file 'dataset_retail.csv' dengan 108 baris.


,id_transaksi,nama_produk,kategori,jumlah_beli,harga_satuan,metode_bayar,kota_pembeli,tanggal_transaksi
0,INV-5000,Laptop Asus ROG,Aksesoris,5.0,20000000.0,KARTU KREDIT,NaN,NaN
1,INV-5001,Keyboard Mechanical,Aksesoris,2.0,NaN,TRANSFER BANK,Surabaya,2026-02-01
2,INV-5002,KEYBOARD MECHANICAL,gadget,10.0,NaN,KARTU KREDIT,bandung,NaN
3,INV-5003,KEYBOARD MECHANICAL,AKSESORIS,3.0,150000.0,e-wallet,JAKARTA,NaN
4,INV-5004,IPHONE 15 PRO,elektronik,2.0,NaN,Transfer Bank,Medan,2026/02/03


**Langkah 2 - Bersihkan teks** (strip + Title Case)

In [8]:
cleaner.bersihkan_teks(KOLOM_TEKS)

for col in KOLOM_TEKS:
    print(f"{col}: {sorted(cleaner.df[col].dropna().unique())}")

# Data kosong harus tetap NaN, bukan berubah jadi teks "Nan"
print(f"\nNaN di kota_pembeli : {cleaner.df['kota_pembeli'].isna().sum()}")
print(f"Teks 'Nan' muncul   : {(cleaner.df['kota_pembeli'] == 'Nan').sum()}")

✓ Kolom teks ['nama_produk', 'kategori', 'metode_bayar', 'kota_pembeli'] telah dibersihkan.
nama_produk: ['Charger Type C', 'Headset Gaming', 'Iphone 15 Pro', 'Keyboard Mechanical', 'Laptop Asus Rog', 'Monitor 24 Inch', 'Mouse Wireless']
kategori: ['Aksesoris', 'Elektronik', 'Gadget']
metode_bayar: ['Cod', 'E-Wallet', 'Kartu Kredit', 'Transfer Bank']
kota_pembeli: ['Bandung', 'Jakarta', 'Medan', 'Surabaya']

NaN di kota_pembeli : 18
Teks 'Nan' muncul   : 0


Catatan: Title Case membuat singkatan ikut berubah (`COD` → `Cod`, `ROG` → `Rog`, `iPhone` → `Iphone`). Ini sesuai instruksi brief, tapi kalau dipakai untuk laporan, singkatan seperti ini sebaiknya dikoreksi dengan mapping tambahan.

**Langkah 3 - Hapus duplikat.** Dijalankan *setelah* bersihkan teks (urutan yang sama dengan hands-on), supaya baris yang hanya beda spasi/kapitalisasi juga ikut terdeteksi.

In [9]:
cleaner.hapus_duplikat()
print(f"Sisa duplikat: {cleaner.df.duplicated().sum()}")

✓ Berhasil menghapus 8 baris duplikat.
Sisa duplikat: 0


**Langkah 4 - Isi nilai kosong pada `jumlah_beli` dengan 0**

In [10]:
cleaner.isi_nilai_kosong("jumlah_beli", nilai_default=0)
print(f"NaN tersisa di jumlah_beli: {cleaner.df['jumlah_beli'].isna().sum()}")
print(f"Tipe data jumlah_beli     : {cleaner.df['jumlah_beli'].dtype}")

✓ 10 nilai kosong pada kolom 'jumlah_beli' diisi dengan 0.
NaN tersisa di jumlah_beli: 0
Tipe data jumlah_beli     : int64


### 5. Validasi Hasil

In [11]:
df_clean = cleaner.df

# Kriteria dari brief
assert df_clean.duplicated().sum() == 0, "masih ada baris duplikat"
assert df_clean["jumlah_beli"].isna().sum() == 0, "jumlah_beli masih ada NaN"
for col in KOLOM_TEKS:
    teks = df_clean[col].dropna()
    assert (teks == teks.str.strip()).all(), f"{col} masih ada spasi berlebih"
    assert (teks == teks.str.title()).all(), f"{col} belum Title Case"
    assert (teks != "Nan").all(), f"{col} punya teks 'Nan' dari NaN"

print("Semua kriteria terpenuhi")
print(f"Sebelum: {df_raw.shape} | Sesudah: {df_clean.shape}")

Semua kriteria terpenuhi
Sebelum: (108, 8) | Sesudah: (100, 8)


In [12]:
df_clean.isna().sum().to_frame("jumlah_nan")

,jumlah_nan
id_transaksi,0
nama_produk,0
kategori,0
jumlah_beli,0
harga_satuan,16
metode_bayar,0
kota_pembeli,17
tanggal_transaksi,21


Kolom di luar cakupan tugas yang masih perlu perhatian kalau data ini dipakai lebih lanjut:
- `harga_satuan` masih punya NaN (tidak diimputasi karena brief hanya meminta `jumlah_beli`).
- `tanggal_transaksi` formatnya campur (`2026-02-01`, `2026/02/03`, `04-02-2026`) dan sebagian kosong.
- `kota_pembeli` masih ada yang kosong.

### 6. Ekspor Data Bersih

In [13]:
cleaner.simpan_hasil("dataset_retail_clean.csv")
pd.read_csv("dataset_retail_clean.csv").head()

✓ Data bersih berhasil disimpan ke: 'dataset_retail_clean.csv' (100 baris).


,id_transaksi,nama_produk,kategori,jumlah_beli,harga_satuan,metode_bayar,kota_pembeli,tanggal_transaksi
0,INV-5000,Laptop Asus Rog,Aksesoris,5,20000000.0,Kartu Kredit,NaN,NaN
1,INV-5001,Keyboard Mechanical,Aksesoris,2,NaN,Transfer Bank,Surabaya,2026-02-01
2,INV-5002,Keyboard Mechanical,Gadget,10,NaN,Kartu Kredit,Bandung,NaN
3,INV-5003,Keyboard Mechanical,Aksesoris,3,150000.0,E-Wallet,Jakarta,NaN
4,INV-5004,Iphone 15 Pro,Elektronik,2,NaN,Transfer Bank,Medan,2026/02/03
